# 01 — Exploratory Data Analysis & Preprocessing
Credit Risk Predictor: load, profile, visualize, and prepare for modeling

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

plt.rcParams['figure.dpi'] = 120
%matplotlib inline

In [ ]:
df = pd.read_csv('../data/credit_risk_data.csv')
print(f'Shape: {df.shape}')
print(f'Default rate: {df.loan_status.eq("Default").mean():.2%}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## Target Distribution

In [ ]:
df['loan_status'].value_counts().plot(kind='bar', color=['#10b981', '#ef4444'])
plt.title('Loan Status Distribution')
plt.ylabel('Count')
plt.xticks(rotation=0);

## Feature Distributions

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(14, 12))
features = [c for c in df.columns if c not in ('loan_status', 'num_delinquencies')]
axes = axes.flatten()
for i, col in enumerate(features):
    axes[i].hist(df[col], bins=40, edgecolor='white', color='#3b82f6')
    axes[i].set_title(col)
plt.tight_layout();

## Default Rate by Feature (Binned)

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(14, 12))
axes = axes.flatten()
for i, col in enumerate(features):
    bins = pd.qcut(df[col], q=10, duplicates='drop')
    default_rate = df.groupby(bins, observed=False)['loan_status'].apply(lambda x: (x == 'Default').mean())
    default_rate.plot(kind='line', marker='o', ax=axes[i], color='#ef4444')
    axes[i].set_title(f'Default Rate by {col}')
    axes[i].set_ylabel('Default Rate')
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout();

## Correlation Matrix

In [ ]:
corr = df.drop(columns=['loan_status']).corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix');

### Drop Redundant Features
`num_delinquencies` correlates at 0.84 with `delinquent_history`. Keep `delinquent_history`.

In [ ]:
X = df.drop(columns=['loan_status', 'num_delinquencies'])
y = (df['loan_status'] == 'Default').astype(int)
print(f'Features ({len(X.columns)}): {list(X.columns)}')

## Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train default rate: {y_train.mean():.2%}')
print(f'Test default rate: {y_test.mean():.2%}')

## Scaling — Fit on Train Only

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Scaler fitted on train, applied to train + test')

## SMOTE — Apply to Train Only

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
print(f'Before SMOTE: {y_train.value_counts().to_dict()}')
print(f'After SMOTE:  {pd.Series(y_train_res).value_counts().to_dict()}')

## Save Scaler

In [ ]:
import joblib
joblib.dump(scaler, '../models/scaler.pkl')
print('Scaler saved to ../models/scaler.pkl')